# Checkpoint shoot-out — which `best.pt` actually generates the right movements

Compares **`checkpoints_1y`, `checkpoints_3y`, `checkpoints_7y`, `checkpoints_all`**
(`best.pt` each) by running the **exact production selection logic in `functions.py`**
over a common set of held-out queries built from the raw `data/*.csv` H1 candles, then
scoring how close each model's generated 50-candle trajectories are to what actually happened.

### How it works

1. **Queries** — random (symbol, bar) positions across all 49 symbols. Every position is
   **excluded if it sits within `HOLDOUT_MARGIN_BARS` of any query used to build
   `candles_1y / candles_3y / candles_7y / candles_train`**, so no checkpoint has been
   trained on the future we score it against. All four models see *identical* inputs.
2. **Analogs** — retrieved with the production scanner: `vectorizer/Vectorizer.py`'s
   `create_candle_vectors` (OHLC / mean-close → per-column z-score → flatten → L2) plus
   cosine top-K over every sliding window of every symbol, with a strict causality mask
   (an analog's 50-bar reaction must have completed **before** the query's last bar).
3. **Generation** — `functions.generate_detailed`'s pipeline, reproduced step-for-step so
   the raw fan is also accessible (a parity cell asserts the returned candles are bit-identical
   to `generate_detailed`'s). Same `torch.manual_seed` per sample for every model → paired comparison.
4. **Scoring** — two families:
   * **fan-level** (no oracle): CRPS, energy score, median-path RMSE, calibration, direction — how good the *distribution* is.
   * **product-level** (the oracle DTW selection `functions.py` actually uses when
     `forward_candle_data` is present): DTW of the surfaced scenarios vs the realised path.
   * plus **teacher-forced NLL** of the true future, converted into common units so it is
     comparable across checkpoints (each `best.pt` carries its own normalization stats).

### Running it

Run top to bottom. Everything is configured in the next cell. Defaults:
`N_QUERIES = 60`, `NUM_SAMPLES = 64` trajectories, `QUERY_LEN = 50`, `K = 50` analogs
→ about **3–4 min** on MPS (≈3 s of loading, ~10 s of scanning, then ≈2.6 s per query across all four models).
Nothing here writes to the repo except `eval_out/` (results snapshot).

In [ ]:
# ───────────────────────── configuration ─────────────────────────
from pathlib import Path
import os, sys, time, json, math, types, glob, datetime as dt

ROOT = Path.cwd()                      # run the notebook from the repo root
DATA_DIR  = ROOT / "data"              # raw H1 CSVs
OUT_DIR   = ROOT / "eval_out"          # results snapshots land here

# the four checkpoints under test  (name -> path)
CKPT_PATHS = {
    "1y":  "checkpoints_1y/best.pt",
    "3y":  "checkpoints_3y/best.pt",
    "7y":  "checkpoints_7y/best.pt",
    "all": "checkpoints_all/best.pt",
}
MODELS = list(CKPT_PATHS)

# ---- service constants that functions.py imports from `model` ----------------
# (the service's model.py is not in this repo; set these to the production values)
HORIZON      = 50      # structural — the model always emits 50 candles
NUM_SAMPLES  = 64      # trajectories sampled per request
TEMPERATURE  = 1.0
SIGMA_SCALE  = 1.0
MIN_WINDOW   = 10      # shortest query build_inputs accepts

# ---- evaluation setup --------------------------------------------------------
QUERY_LEN    = 50      # bars in the scanned query window (1..50, 60..100)
N_ANALOGS    = 50      # K handed to the model (matches cfg.val_k)
K_SCAN       = 200     # retrieve this many, slice K from the top (enables the K sweep)
N_QUERIES    = 60      # held-out queries to evaluate (each costs ~4 s across 4 models)
SEED         = 7       # query sampling + per-sample torch seed
MIN_QUERY_DATE = "2013-01-01"   # leave the early years as analog bank, not queries

# leakage guard: drop any candidate query within this many bars of a query that
# built ANY of the four training datasets (their targets would overlap ours).
HOLDOUT_MARGIN_BARS = 150       # = QUERY_LEN + HORIZON, so neither window touches
TRAIN_META_DIRS = ["candles_1y", "candles_3y", "candles_7y", "candles_train"]

# same-symbol analogs whose window ends within this many bars of the query are
# dropped as near-duplicates (0 = production behaviour, keep everything causal)
EXCLUDE_SELF_BARS = 0

PRIMARY_METRIC = "crps_step"    # metric the win-rate matrix is computed on
# metrics averaged into the composite rank
HEADLINE = ["nll", "crps_step", "energy", "rmse_median", "dtw_top1"]

import numpy as np, torch
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")

OUT_DIR.mkdir(exist_ok=True)
print("root:", ROOT, "\ndevice:", DEVICE)
for n, p in CKPT_PATHS.items():
    print(f"  {n:4s} {p:28s} {'OK' if (ROOT/p).exists() else 'MISSING'}")

## 1 · Wire up `functions.py`

`functions.py` is the service module: it pulls its constants from a `model` module that lives in
the service repo, and `data_model.py` needs `pydantic`. Neither is here, so both are stubbed in
memory — **no repo file is modified**. `functions.py` itself is imported and used verbatim.

In [ ]:
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

# pydantic shim (only if the real thing is absent) — data_model uses BaseModel purely
# as a keyword-argument container, so a passthrough class is behaviourally identical.
try:
    import pydantic                                  # noqa: F401
except ImportError:
    _pyd = types.ModuleType("pydantic")
    class _BaseModel:
        def __init__(self, **kw):
            for k, v in kw.items(): setattr(self, k, v)
    _pyd.BaseModel = _BaseModel
    sys.modules["pydantic"] = _pyd
    print("pydantic not installed -> using in-memory shim")

# the service-side `model` module functions.py imports its constants from
_svc = types.ModuleType("model")
_svc.CKPT_PATHS = dict(CKPT_PATHS)
_svc.DEFAULT_MODEL_VERSION = MODELS[0]
_svc.DEVICE = DEVICE
_svc.HORIZON = HORIZON
_svc.MIN_WINDOW = min(MIN_WINDOW, QUERY_LEN)
_svc.NUM_SAMPLES = NUM_SAMPLES
_svc.SIGMA_SCALE = SIGMA_SCALE
_svc.TEMPERATURE = TEMPERATURE
sys.modules["model"] = _svc                          # shadows the training-side model.py

import functions as FN
from data_model import CandleModel, SimilarPatternModel, GenerateModel
from candle_model import ohlc_to_features, features_to_ohlc, vol_scale

import warnings; warnings.filterwarnings("ignore", message=".*enable_nested_tensor.*")
print("functions.py loaded — HORIZON", FN.HORIZON, "| NUM_SAMPLES", FN.NUM_SAMPLES,
      "| DTW top fraction", FN.DTW_TOP_FRACTION, "| max scenarios", FN.DTW_MAX_SCENARIOS)

## 2 · What is in each checkpoint

Every `best.pt` carries its own `cfg` and normalization `stats`. `best_val` is **not** comparable
across rows — different datasets mean different target distributions — which is exactly why this
notebook re-scores all four on one common set.

In [ ]:
def _fmt(x, n=4):
    return f"{x:.{n}f}" if isinstance(x, float) else str(x)

def print_table(headers, rows, aligns=None):
    cols = [len(h) for h in headers]
    srows = [[_fmt(c) if not isinstance(c, str) else c for c in r] for r in rows]
    for r in srows:
        for i, c in enumerate(r): cols[i] = max(cols[i], len(c))
    aligns = aligns or ["<"] + [">"] * (len(headers) - 1)
    line = "  ".join(f"{h:{a}{w}}" for h, w, a in zip(headers, cols, aligns))
    print(line); print("─" * len(line))
    for r in srows:
        print("  ".join(f"{c:{a}{w}}" for c, w, a in zip(r, cols, aligns)))

rows = []
CKPT_INFO = {}
for name, path in CKPT_PATHS.items():
    ck = torch.load(ROOT / path, map_location="cpu")
    cfg, st = ck["cfg"], ck["stats"]
    CKPT_INFO[name] = {"epoch": ck["epoch"], "step": ck["step"], "best_val": ck["best_val"],
                       "data_dir": cfg["data_dir"], "stats": st}
    rows.append([name, cfg["data_dir"], f"{ck['epoch']+1}/{cfg['epochs']}", str(ck["step"]),
                 f"{ck['best_val']:.3f}",
                 "[" + ", ".join(f"{v:.3f}" for v in st["std"]) + "]"])
    del ck
print_table(["model", "trained on", "epochs", "opt steps", "best val NLL*", "feature std (gap, body, up, low)"], rows)
print("\n* val NLL is only meaningful within one dataset — see the common-units NLL computed below.")

## 3 · Load the raw candles

All 49 symbols, H1, from `data/`. Rows are sorted, de-duplicated on timestamp, and any bar that is
non-finite or non-positive is dropped (the log-space candle features require strictly positive prices).

In [ ]:
def utc(ts):                      # UTC datetime from a unix second stamp
    return dt.datetime.fromtimestamp(int(ts), dt.timezone.utc)

def load_symbol(path):
    with open(path) as f:
        head = f.readline().strip().split(",")
        idx = [head.index(c) for c in ("timestamp", "open", "high", "low", "close")]
        rows = [l.split(",") for l in f]
    a = np.array([[r[i] for i in idx] for r in rows], dtype=np.float64)
    a = a[np.argsort(a[:, 0], kind="stable")]
    _, u = np.unique(a[:, 0], return_index=True)
    a = a[np.sort(u)]
    ts, ohlc = a[:, 0].astype(np.int64), a[:, 1:5]
    ok = np.isfinite(ohlc).all(1) & (ohlc > 0).all(1)
    return ts[ok], ohlc[ok]

t0 = time.time()
SYMS = {Path(p).stem: load_symbol(p) for p in sorted(glob.glob(str(DATA_DIR / "*.csv")))}
NAMES = sorted(SYMS)
n_bars = sum(len(v[0]) for v in SYMS.values())
span = (min(v[0][0] for v in SYMS.values()), max(v[0][-1] for v in SYMS.values()))
print(f"{len(SYMS)} symbols · {n_bars:,} H1 bars · "
      f"{utc(span[0]):%Y-%m-%d} → {utc(span[1]):%Y-%m-%d} "
      f"({time.time()-t0:.1f}s)")

## 4 · The scanner (production parity)

The similarity used to build the training datasets — and used live — is
`vectorizer/Vectorizer.py::create_candle_vectors`: OHLC divided by the window's mean close,
each of the four channels z-scored across the window, flattened, L2-normalized; similarity is the
cosine of two such vectors (which is why scores sit around 0.9). The cell below is a vectorized
re-implementation, checked against the original when `~/Documents/model` is reachable.

In [ ]:
def vectorize(win):
    """(n, W, 4) raw OHLC -> (n, 4W) unit vectors. Parity with create_candle_vectors."""
    win = np.asarray(win, np.float32)
    ref = win[:, :, 3].mean(1)[:, None, None]              # mean close
    f = win / ref
    f = (f - f.mean(1, keepdims=True)) / (f.std(1, keepdims=True) + 1e-8)
    v = f.reshape(len(win), -1)
    nrm = np.linalg.norm(v, axis=1, keepdims=True); nrm[nrm == 0] = 1.0
    return v / nrm

# --- parity check against the production vectorizer (skipped if not on this machine) ---
try:
    sys.path.insert(0, os.path.expanduser("~/Documents/model"))
    from vectorizer.Vectorizer import create_candle_vectors
    class _Col:
        def __init__(s, v): s.values = v
    class _DF(dict):
        def __getitem__(s, k): return _Col(dict.__getitem__(s, k))
    w = SYMS[NAMES[0]][1][5000:5000 + QUERY_LEN]
    ref = create_candle_vectors(_DF(open=w[:, 0], high=w[:, 1], low=w[:, 2], close=w[:, 3]))
    print(f"vectorizer parity vs production: max |diff| = {np.abs(ref - vectorize(w[None])[0]).max():.2e}")
except Exception as e:
    print("production vectorizer not importable, parity check skipped:", type(e).__name__, e)

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

def bank_vectors(sym, W, H):
    """Every window of length W in `sym` that has H bars of reaction after it."""
    ts, oh = SYMS[sym]
    if len(ts) < W + H + 1: return None
    wins = sliding_window_view(oh, (W, 4)).reshape(-1, W, 4)     # window i ends at bar i+W-1
    nb = len(wins) - H
    if nb <= 0: return None
    end_idx = np.arange(W - 1, W - 1 + nb)
    return vectorize(wins[:nb]), end_idx, ts[end_idx + H]        # ts of the last reaction bar

def scan(queries, W, H, K, exclude_self_bars=0, verbose=True):
    """queries: [(symbol, end_bar_index)] -> per query: top-K analogs, score-sorted desc.

    An analog is admissible only if its whole H-bar reaction completed at or before the
    query's last bar — otherwise the model would be handed the future it is asked to predict."""
    qv = np.stack([vectorize(SYMS[s][1][e - W + 1:e + 1][None])[0] for s, e in queries])
    q_ts = np.array([SYMS[s][0][e] for s, e in queries])
    m = len(queries)
    best_sc  = np.full((m, K), -np.inf, np.float32)
    best_sym = np.full((m, K), -1, np.int32)
    best_end = np.zeros((m, K), np.int64)
    t0 = time.time()
    for si, sym in enumerate(NAMES):
        b = bank_vectors(sym, W, H)
        if b is None: continue
        v, end_idx, fut_end_ts = b
        sims = (qv @ v.T).astype(np.float32)                     # (m, nb) cosine
        sims[fut_end_ts[None, :] > q_ts[:, None]] = -np.inf       # causality mask
        if exclude_self_bars:
            for qi, (s2, e2) in enumerate(queries):
                if s2 == sym:
                    sims[qi, np.abs(end_idx - e2) < exclude_self_bars] = -np.inf
        kk = min(K, sims.shape[1])
        top = np.argpartition(-sims, kk - 1, axis=1)[:, :kk]
        cat_sc  = np.concatenate([best_sc,  np.take_along_axis(sims, top, 1)], 1)
        cat_sym = np.concatenate([best_sym, np.full_like(top, si, dtype=np.int32)], 1)
        cat_end = np.concatenate([best_end, end_idx[top]], 1)
        order = np.argsort(-cat_sc, axis=1, kind="stable")[:, :K]
        best_sc  = np.take_along_axis(cat_sc,  order, 1)
        best_sym = np.take_along_axis(cat_sym, order, 1)
        best_end = np.take_along_axis(cat_end, order, 1)
    if verbose:
        print(f"scanned {len(NAMES)} symbols for {m} queries in {time.time()-t0:.1f}s")
    out = []
    for qi in range(m):
        keep = best_sc[qi] > -np.inf
        out.append({"scores": best_sc[qi][keep].astype(np.float64),
                    "syms":  [NAMES[j] for j in best_sym[qi][keep]],
                    "ends":  best_end[qi][keep]})
    return out

## 5 · Held-out query selection

Every dataset the four checkpoints were trained on came from these same CSVs, so "out of sample"
has to be constructed positionally. A candidate query is dropped if **any** training query for that
symbol (across all four `meta.npz` files) sits within `HOLDOUT_MARGIN_BARS` bars — at the default
150 = `QUERY_LEN + HORIZON` neither the query window nor the 50-bar future we score against can
overlap anything a model was trained on.

In [ ]:
trained_ts = {s: [] for s in NAMES}
found = []
for d in TRAIN_META_DIRS:
    p = ROOT / d / "meta.npz"
    if not p.exists(): continue
    m = np.load(p, allow_pickle=True)
    for s, t in zip(m["q_symbol"], (m["q_end_ts"] // 1_000_000_000).astype(np.int64)):
        s = str(s)
        if s in trained_ts: trained_ts[s].append(int(t))
    found.append(f"{d}({len(m['q_symbol'])})")
print("training query sets loaded:", ", ".join(found) or "none found")

ELIGIBLE = {}
min_ts = int(dt.datetime.strptime(MIN_QUERY_DATE, "%Y-%m-%d").timestamp())
tot_ok = tot_bars = 0
for s in NAMES:
    ts, oh = SYMS[s]
    n = len(ts)
    ok = np.zeros(n, bool)
    lo, hi = QUERY_LEN - 1, n - HORIZON - 1
    if hi > lo: ok[lo:hi] = True
    ok &= ts >= min_ts
    if trained_ts[s]:
        hits = np.unique(np.searchsorted(ts, np.array(sorted(set(trained_ts[s])))))
        hits = hits[(hits >= 0) & (hits < n)]
        for i in hits:
            ok[max(0, i - HOLDOUT_MARGIN_BARS): i + HOLDOUT_MARGIN_BARS + 1] = False
    ELIGIBLE[s] = np.flatnonzero(ok)
    tot_ok += len(ELIGIBLE[s]); tot_bars += n
print(f"eligible query positions: {tot_ok:,} of {tot_bars:,} bars "
      f"({tot_ok/tot_bars:.0%} survive the ±{HOLDOUT_MARGIN_BARS}-bar hold-out + date filter)")

rng = np.random.default_rng(SEED)
pool = [s for s in NAMES if len(ELIGIBLE[s]) > 0]
QUERIES = []
i = 0
while len(QUERIES) < N_QUERIES:                       # round-robin so no symbol dominates
    s = pool[i % len(pool)]; i += 1
    e = int(rng.choice(ELIGIBLE[s]))
    if (s, e) not in QUERIES: QUERIES.append((s, e))
print(f"\nsampled {len(QUERIES)} queries across {len(set(s for s,_ in QUERIES))} symbols")
print("  e.g.", ", ".join(f"{s}@{utc(SYMS[s][0][e]):%Y-%m-%d}"
                          for s, e in QUERIES[:6]), "...")

In [ ]:
ANALOGS = scan(QUERIES, QUERY_LEN, HORIZON, K_SCAN, EXCLUDE_SELF_BARS)
sc = np.concatenate([a["scores"][:N_ANALOGS] for a in ANALOGS])
print(f"top-{N_ANALOGS} similarity: median {np.median(sc):.3f}  "
      f"p10 {np.percentile(sc,10):.3f}  p90 {np.percentile(sc,90):.3f}  "
      f"(training datasets reported a median around 0.88–0.91)")
print("analogs retrieved per query:", np.unique([len(a['scores']) for a in ANALOGS]))

In [ ]:
def to_candles(ts, ohlc):
    return [CandleModel(o=float(o), h=float(h), l=float(l), c=float(c), timestamp=int(t))
            for t, (o, h, l, c) in zip(ts, ohlc)]

def make_request(sym, end, analogs, W=QUERY_LEN, H=HORIZON, K=N_ANALOGS):
    """One GenerateModel exactly as the service would receive it, plus the realised
    future in forward_candle_data (which switches functions.py to DTW selection)."""
    ts, oh = SYMS[sym]
    pats = []
    for s2, e2, score in zip(analogs["syms"][:K], analogs["ends"][:K], analogs["scores"][:K]):
        ts2, oh2 = SYMS[s2]
        pats.append(SimilarPatternModel(
            pattern_candles=to_candles(ts2[e2 - W + 1:e2 + 1], oh2[e2 - W + 1:e2 + 1]),
            reaction_candles=to_candles(ts2[e2 + 1:e2 + 1 + H], oh2[e2 + 1:e2 + 1 + H]),
            similarity_score=float(score), end_timestamp=float(ts2[e2])))
    return GenerateModel(
        scanned_candles=to_candles(ts[end - W + 1:end + 1], oh[end - W + 1:end + 1]),
        similar_patterns=pats,
        q_anchor=float(oh[end, 3]),
        forward_candle_data=to_candles(ts[end + 1:end + 1 + H], oh[end + 1:end + 1 + H]))

REQUESTS, KEPT = [], []
_cfg = FN.load_model(FN.resolve_ckpt_path(MODELS[0])).cfg
for (s, e), a in zip(QUERIES, ANALOGS):
    r = make_request(s, e, a)
    try:
        FN.build_inputs(r, _cfg)                     # same validation the service runs
        REQUESTS.append(r); KEPT.append((s, e))
    except FN.BatchError as err:
        print(f"  dropped {s}@{e}: {err}")
print(f"{len(REQUESTS)} requests ready "
      f"({len(REQUESTS[0].similar_patterns)} analogs, {len(REQUESTS[0].scanned_candles)}-bar query, "
      f"{len(REQUESTS[0].forward_candle_data)}-bar realised future)")

## 6 · Reproduce `generate_detailed`, keep the raw fan

`generate_detailed` returns only the *selected* scenarios. Distributional scoring needs the whole
fan, so the cell below re-walks its body (`build_inputs` → anchor rescale → `torch.manual_seed` →
`gen.generate` → `choose_scenarios_dtw` → `score_against_actual`) using `functions.py`'s own
functions, and keeps everything. The parity cell after it asserts the surfaced candles are
**bit-identical** to what the service would return.

In [ ]:
def run_generation(req, version, seed=0, n_traj=NUM_SAMPLES, temperature=TEMPERATURE):
    loaded = FN.load_model(FN.resolve_ckpt_path(version))
    prep = FN.build_inputs(req, loaded.cfg)
    q_pat = prep["q_pat"]
    anchor = float(req.q_anchor) if req.q_anchor else float(q_pat[-1, 3])
    if not np.isfinite(anchor) or anchor <= 0: anchor = float(q_pat[-1, 3])
    q_anchored = q_pat * (anchor / float(q_pat[-1, 3]))

    torch.manual_seed(seed)
    t0 = time.time()
    trajs = loaded.gen.generate(q_anchored, prep["patterns"], prep["futures"], prep["scores"],
                                n_traj=n_traj, temperature=temperature, sigma_scale=SIGMA_SCALE)
    gen_s = time.time() - t0
    paths = np.ascontiguousarray(trajs[:, :, 3])

    actual = FN.candles_to_ohlc(req.forward_candle_data)
    FN._validate_ohlc(actual, "forward_candle_data")
    scen, meta = FN.choose_scenarios_dtw(paths, anchor, actual_closes=actual[:, 3])
    closest = FN.score_against_actual(scen, actual[:, 3], anchor)
    return dict(version=version, trajs=trajs, paths=paths, scen=scen, meta=meta,
                closest=closest, anchor=anchor, actual=actual, q_anchored=q_anchored,
                prep=prep, loaded=loaded, gen_s=gen_s)

@torch.no_grad()
def teacher_forced_nll(r):
    """NLL of the REAL future under the model's mixture head, in common units.

    Each checkpoint standardizes features with its own (mu, sd), so the raw NLL lives in a
    different space per model. Changing variables back to the un-normalized log-feature space
    (add sum(log sd) for the standardization, 4*log(s) for the per-sample vol scale — the
    latter is identical across checkpoints) makes the four numbers directly comparable.
    Lower is better; this is the same quantity train.py optimizes."""
    gen, cfg = r["loaded"].gen, r["loaded"].gen.cfg
    batch, s, last_close = gen._prep(r["q_anchored"], r["prep"]["patterns"],
                                     r["prep"]["futures"], r["prep"]["scores"])
    tgt = ohlc_to_features(r["actual"], prev_close=last_close) / s
    tgt = np.clip((tgt - gen.mu) / gen.sd, -cfg.feat_clip, cfg.feat_clip)
    tgt = torch.from_numpy(tgt.astype(np.float32))[None].to(gen.device)
    memory, mem_pad = gen.model.encode(batch)
    logits, mean, log_sigma = gen.model._decode(tgt[:, :-1], memory, mem_pad)
    t = tgt.unsqueeze(2)
    lp = -0.5 * ((t - mean) ** 2 * torch.exp(-2.0 * log_sigma) + 2.0 * log_sigma + math.log(2 * math.pi))
    lp = lp.sum(-1) + torch.log_softmax(logits, dim=-1)
    nll_z = float(-torch.logsumexp(lp, dim=-1).mean())
    return nll_z + float(np.log(gen.sd).sum()) + 4.0 * float(np.log(s))

In [ ]:
# parity: our reproduction vs the real generate_detailed, same seed
probe = REQUESTS[0]
for v in MODELS:
    r = run_generation(probe, v, seed=0)
    out = FN.generate_detailed(probe, model_version=v, seed=0)
    got = np.array([[c.o, c.h, c.l, c.c] for c in out["groups"].groups[0].candles])
    mine = r["trajs"][r["scen"][0].idx]
    assert np.array_equal(got, mine), f"{v}: reproduction diverged from generate_detailed"
    assert len(out["groups"].groups) == len(r["scen"])
    print(f"  {v:4s} identical — {len(r['scen'])} scenarios surfaced, "
          f"top DTW {r['scen'][0].dtw:.4f}, best RMSE {min(s.rmse for s in r['scen']):.4f}, "
          f"{r['gen_s']:.1f}s")
print("\nparity OK — the metrics below score exactly what the service would return.")

## 7 · Metrics

Everything is computed on **log returns against the anchor**, `y_t = log(close_t / anchor)`, so a
JPY cross and BTC contribute on the same scale.

**Fan-level — scores the distribution the model actually produces (no peeking):**

| metric | what it says | good |
|---|---|---|
| `nll` | teacher-forced NLL of the true future, common units | low |
| `crps_step` | CRPS of the ensemble vs the realised close, averaged over the 50 steps. Proper score: rewards being right *and* being honestly spread | low |
| `energy` | energy score over the whole 50-dim path (multivariate CRPS), per-step normalized — penalizes paths that are right pointwise but wrong in shape | low |
| `rmse_median` | RMSE of the fan's pointwise median vs the realised path | low |
| `mae_terminal` | \|median 50-bar return − realised 50-bar return\| | low |
| `cover90` / `cover50` | share of steps the realised close sat inside the 5–95% / 25–75% band | ≈0.90 / ≈0.50 |
| `brier_up` | Brier score of P(upward move), labelled by the same max-vs-min excursion rule `functions.py` uses | low |
| `dir_med` | median path's direction matched reality | high |
| `vol_ratio` | generated realized vol ÷ actual realized vol | ≈1.0 |
| `dtw_median` | DTW (z-normalized, Sakoe-Chiba band 5) of the median path vs reality | low |

**Product-level — the scenarios `functions.py` actually surfaces** (with `forward_candle_data`
present it ranks the fan by DTW against the realised path, so these are oracle-selected and
measure *"does the model's fan contain the right movement, and does the shipped list lead with it"*):

| metric | what it says | good |
|---|---|---|
| `dtw_top1` | DTW of the best-matching surfaced scenario | low |
| `dtw_sel_mean` | mean DTW across every surfaced scenario | low |
| `rmse_best` | best RMSE among surfaced scenarios (÷ anchor) | low |
| `label_top1` | the lead scenario's UPWARD/DOWNWARD label matched reality | high |
| `n_groups` | how many scenarios were surfaced | — |

In [ ]:
def crps_ensemble(ens, x):
    """Fair (ensemble-size-unbiased) CRPS of sample `ens` against observation `x`.

    The naive 0.5*mean|y_i-y_j| spread term is biased low for small ensembles, which would
    hand an advantage to whichever forecast has more members. The n(n-1) normalization
    removes that, so a 64-path model fan, a 50-member analog ensemble and a single
    deterministic path are all scored on the same footing."""
    n = len(ens)
    t1 = float(np.abs(ens - x).mean())
    if n < 2: return t1                                  # deterministic forecast: CRPS = MAE
    return t1 - float(np.abs(ens[:, None] - ens[None, :]).sum() / (2.0 * n * (n - 1)))

def energy_score(Y, x):
    """Fair multivariate CRPS (beta=1) of the path ensemble Y (N,T) against x (T,)."""
    n, T = Y.shape
    d1 = float(np.linalg.norm(Y - x[None], axis=1).mean())
    if n < 2: return d1 / math.sqrt(T)
    d2 = float(np.linalg.norm(Y[:, None] - Y[None], axis=2).sum() / (n * (n - 1)))
    return (d1 - 0.5 * d2) / math.sqrt(T)

def excursion_label(closes, anchor):
    """functions.py's rule: a path that spikes up then round-trips is still an UPWARD move."""
    return "UPWARD" if (closes.max() / anchor - 1.0) >= -(closes.min() / anchor - 1.0) else "DOWNWARD"

def fan_metrics(paths, actual_closes, anchor):
    """Score ANY ensemble of close paths (N, 50) against the realised closes.

    Kept separate from the model-specific metrics so the baselines in section 11 are
    scored by exactly this code — no second implementation to drift out of sync."""
    A = anchor
    Y = np.log(np.asarray(paths, np.float64) / A)   # (N, 50) forecast log paths
    x = np.log(np.asarray(actual_closes, np.float64) / A)
    N, T = Y.shape

    crps_t = np.array([crps_ensemble(Y[:, t], x[t]) for t in range(T)])
    med = np.median(Y, axis=0)
    q05, q25, q75, q95 = np.percentile(Y, [5, 25, 75, 95], axis=0)

    act_lab = excursion_label(np.asarray(actual_closes, np.float64), A)
    p_up = float(np.mean([excursion_label(A * np.exp(y), A) == "UPWARD" for y in Y]))
    dv = np.diff(Y, axis=1).std(axis=1)
    av = float(np.diff(x).std())

    m = dict(
        crps_step    = float(crps_t.mean()),
        energy       = energy_score(Y, x),
        rmse_median  = float(np.sqrt(((med - x) ** 2).mean())),
        mae_terminal = float(abs(med[-1] - x[-1])),
        cover90      = float(np.mean((x >= q05) & (x <= q95))),
        cover50      = float(np.mean((x >= q25) & (x <= q75))),
        brier_up     = float((p_up - float(act_lab == "UPWARD")) ** 2),
        dir_med      = float(np.sign(med[-1]) == np.sign(x[-1])),
        vol_ratio    = float(np.median(dv) / (av + 1e-12)),
        dtw_median   = float(FN.dtw_distance(FN.znorm(med), FN.znorm(x))),
        rmse_bestfan = float(np.min(np.sqrt(((Y - x[None]) ** 2).mean(axis=1)))),
        pit_terminal = float((np.sum(Y[:, -1] < x[-1]) + 0.5 * np.sum(Y[:, -1] == x[-1])) / N),
    )
    return m, crps_t

def compute_metrics(r):
    """fan_metrics + what only the model has: its own NLL and the surfaced scenarios."""
    m, crps_t = fan_metrics(r["paths"], r["actual"][:, 3], r["anchor"])
    sel, A = r["scen"], r["anchor"]
    act_lab = excursion_label(r["actual"][:, 3], A)
    m.update(
        nll          = teacher_forced_nll(r),
        dtw_top1     = float(sel[0].dtw),
        dtw_sel_mean = float(np.mean([s.dtw for s in sel])),
        rmse_best    = float(min(s.rmse for s in sel)),
        label_top1   = float(excursion_label(sel[0].path, A) == act_lab),
        n_groups     = float(len(sel)),
        gen_s        = r["gen_s"],
    )
    return m, crps_t

# (key, label, better: True=lower / False=higher / None=hit the target, aggregator)
# error metrics are heavy-tailed -> median; rates and flags are proportions -> mean
METRIC_SPECS = [
    ("nll",          "teacher-forced NLL",       True,  "median"),
    ("crps_step",    "CRPS per step",            True,  "median"),
    ("energy",       "energy score",             True,  "median"),
    ("rmse_median",  "median-path RMSE",         True,  "median"),
    ("dtw_top1",     "DTW, best scenario",       True,  "median"),
    ("dtw_sel_mean", "DTW, surfaced mean",       True,  "median"),
    ("dtw_median",   "DTW, median path",         True,  "median"),
    ("rmse_best",    "RMSE, best scenario",      True,  "median"),
    ("rmse_bestfan", "RMSE, best of fan",        True,  "median"),
    ("mae_terminal", "terminal MAE",             True,  "median"),
    ("brier_up",     "Brier, P(up)",             True,  "mean"),
    ("dir_med",      "direction hit (median)",   False, "mean"),
    ("label_top1",   "lead label correct",       False, "mean"),
    ("cover90",      "90% band coverage",        None,  "mean"),
    ("cover50",      "50% band coverage",        None,  "mean"),
    ("vol_ratio",    "vol ratio (gen/actual)",   None,  "median"),
    ("n_groups",     "scenarios surfaced",       None,  "mean"),
]
TARGETS = {"cover90": 0.90, "cover50": 0.50, "vol_ratio": 1.0}
print("metrics defined:", len(METRIC_SPECS))

## 8 · Run every checkpoint over every query

Same query, same analogs, same seed per sample → paired comparison. Progress prints as it goes.

In [ ]:
RESULTS   = {v: [] for v in MODELS}      # per-sample metric dicts
CRPS_CURVE = {v: [] for v in MODELS}     # per-sample (50,) CRPS by horizon step
FANS      = {v: [] for v in MODELS}      # per-sample (N,50,4) trajectories, for the plots
SCEN      = {v: [] for v in MODELS}      # per-sample selected scenarios

t_start = time.time()
for i, req in enumerate(REQUESTS):
    for v in MODELS:
        r = run_generation(req, v, seed=SEED + i)
        m, crps_t = compute_metrics(r)
        RESULTS[v].append(m); CRPS_CURVE[v].append(crps_t)
        FANS[v].append(r["trajs"].astype(np.float32)); SCEN[v].append(r["scen"])
    if (i + 1) % 5 == 0 or i == len(REQUESTS) - 1:
        el = time.time() - t_start
        print(f"  {i+1:3d}/{len(REQUESTS)} queries · {el:5.0f}s elapsed · "
              f"{el/(i+1):.1f}s per query · eta {el/(i+1)*(len(REQUESTS)-i-1):5.0f}s")

for v in MODELS:
    CRPS_CURVE[v] = np.array(CRPS_CURVE[v])
M = {v: {k: np.array([m[k] for m in RESULTS[v]]) for k in RESULTS[v][0]} for v in MODELS}
print(f"\ndone — {len(REQUESTS)} queries x {len(MODELS)} models in {time.time()-t_start:.0f}s")

In [ ]:
# snapshot the run so the tables/plots can be rebuilt without regenerating
stamp = dt.datetime.now().strftime("%Y%m%d-%H%M%S")
snap = OUT_DIR / f"ckpt_comparison_{stamp}.npz"
np.savez_compressed(
    snap,
    config=json.dumps(dict(query_len=QUERY_LEN, n_analogs=N_ANALOGS, n_queries=len(REQUESTS),
                           num_samples=NUM_SAMPLES, temperature=TEMPERATURE, sigma_scale=SIGMA_SCALE,
                           seed=SEED, holdout_bars=HOLDOUT_MARGIN_BARS, device=DEVICE,
                           queries=[[s, int(e)] for s, e in KEPT], models=MODELS)),
    **{f"{v}__{k}": M[v][k] for v in MODELS for k in M[v]},
    **{f"{v}__crps_curve": CRPS_CURVE[v] for v in MODELS})
print("saved", snap)

## 9 · Results

In [ ]:
rows = []
for key, label, lower, how in METRIC_SPECS:
    f = np.median if how == "median" else np.mean
    vals = {v: float(f(M[v][key])) for v in MODELS}
    if lower is True:    best, better = min(vals, key=vals.get), "lower"
    elif lower is False: best, better = max(vals, key=vals.get), "higher"
    elif key in TARGETS: best, better = min(vals, key=lambda v: abs(vals[v] - TARGETS[key])), f"≈{TARGETS[key]}"
    else:                best, better = None, "—"
    rows.append([label, how] + [f"{vals[v]:.4f}" + (" *" if v == best else "  ") for v in MODELS] + [better])
print(f"Aggregated over {len(REQUESTS)} held-out queries   (* = best in row)\n")
print_table(["metric", "stat"] + MODELS + ["better"], rows)

In [ ]:
# paired per-query ranking — a couple of wild queries cannot dominate the verdict
DIRECTION = {k: l for k, _, l, _a in METRIC_SPECS}

def rank_tensor(keys):
    """(metrics, models, queries) of 1..4 ranks, 1 = best on that metric for that query."""
    T = np.zeros((len(keys), len(MODELS), len(REQUESTS)))
    for j, key in enumerate(keys):
        vals = np.stack([M[v][key] for v in MODELS])            # (models, queries)
        if DIRECTION[key] is None:                              # target metric -> distance to target
            vals = np.abs(vals - TARGETS.get(key, 0.0))
        sgn = -1.0 if DIRECTION[key] is False else 1.0          # higher-is-better flips
        T[j] = np.argsort(np.argsort(sgn * vals, axis=0, kind="stable"), axis=0) + 1
    return T

RT = rank_tensor(HEADLINE)                    # (metrics, models, queries)
mean_rank = RT.mean(axis=(0, 2))
print(f"COMPOSITE — rank across {HEADLINE},")
print(f"pooled over {len(REQUESTS)} queries x {len(HEADLINE)} metrics (1.00 = best every time)\n")
print_table(["model", "mean rank", *[f"ranked {i+1}" for i in range(len(MODELS))]],
            [[v, f"{mean_rank[i]:.2f}",
              *[f"{np.mean(RT[:, i, :] == j + 1):.0%}" for j in range(len(MODELS))]]
             for i, v in enumerate(MODELS)])

print(f"\nHEAD-TO-HEAD on {PRIMARY_METRIC} — share of queries where the row model beats the column model\n")
if DIRECTION[PRIMARY_METRIC] is None:
    prim = {v: np.abs(M[v][PRIMARY_METRIC] - TARGETS.get(PRIMARY_METRIC, 0.0)) for v in MODELS}
    better_is_lower = True
else:
    prim = {v: M[v][PRIMARY_METRIC] for v in MODELS}
    better_is_lower = DIRECTION[PRIMARY_METRIC] is True
rows = []
for a in MODELS:
    row = [a]
    for b in MODELS:
        row.append("—" if a == b else
                   f"{np.mean(prim[a] < prim[b]) if better_is_lower else np.mean(prim[a] > prim[b]):.0%}")
    rows.append(row)
print_table(["", *MODELS], rows)

winner = MODELS[int(np.argmin(mean_rank))]
print(f"\n>>> best overall: {winner}  (mean rank {mean_rank.min():.2f}, "
      f"trained on {CKPT_INFO[winner]['data_dir']}, epoch {CKPT_INFO[winner]['epoch']+1})")

### Charts

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# categorical palette (validated: all-pairs CVD ΔE 9.2, normal-vision 16.3, light surface)
MC = {"1y": "#2a78d6", "3y": "#eb6834", "7y": "#1baf7a", "all": "#4a3aa7"}
MC = {v: MC.get(v, "#2a78d6") for v in MODELS}
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e5e4e0", "#fcfcfb"

def tint(hexcolor, f):
    c = np.array([int(hexcolor[i:i+2], 16) for i in (1, 3, 5)]) / 255
    return tuple(c + (1 - c) * f)

def style(ax, ygrid=True):
    ax.set_facecolor(SURFACE)
    if ygrid: ax.grid(axis="y", color=GRID, lw=0.7, zorder=0)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=8)

PANELS = [("nll", "Teacher-forced NLL", "lower is better"),
          ("crps_step", "CRPS per step", "lower is better"),
          ("energy", "Energy score", "lower is better"),
          ("rmse_median", "Median-path RMSE", "lower is better"),
          ("dtw_top1", "DTW of best surfaced scenario", "lower is better"),
          ("cover90", "90% band coverage", "target 0.90")]

AGGS = {k: (np.median if how == "median" else np.mean) for k, _, _d, how in METRIC_SPECS}

fig, axes = plt.subplots(2, 3, figsize=(12.5, 6.6), facecolor="white")
for ax, (key, title, note) in zip(axes.ravel(), PANELS):
    vals = [float(AGGS[key](M[v][key])) for v in MODELS]
    y = np.arange(len(MODELS))
    lo, hi = min(vals), max(vals)
    zero_based = lo >= 0            # bars need a real zero baseline; otherwise dot plot
    if zero_based:
        ax.barh(y, vals, height=0.6, color=[MC[v] for v in MODELS], zorder=3)
        left, right = 0.0, hi + 0.28 * (hi or 1)
    else:
        pad = (hi - lo) or abs(hi) or 1.0
        left, right = lo - 0.35 * pad, hi + 0.45 * pad
        for i, v in enumerate(MODELS):
            ax.plot([left, vals[i]], [i, i], color=tint(MC[v], 0.72), lw=2, zorder=2,
                    solid_capstyle="butt")
            ax.plot([vals[i]], [i], "o", ms=9, color=MC[v], zorder=3)
    for i, val in enumerate(vals):
        ax.text(val + 0.045 * (right - left), i, f"{val:,.3g}", va="center", ha="left",
                fontsize=9, color=INK)
    if key in TARGETS:
        ax.axvline(TARGETS[key], color=MUTED, lw=1.2, ls=(0, (4, 3)), zorder=4)
        left = min(left, TARGETS[key] - 0.06 * (right - left))
    ax.set_xlim(left, right)
    ax.set_yticks(y); ax.set_yticklabels(MODELS, fontsize=9.5, color=INK); ax.invert_yaxis()
    ax.set_title(title, fontsize=10.5, color=INK, loc="left", pad=8)
    ax.set_xlabel(note, fontsize=8, color=MUTED, loc="left")
    style(ax, ygrid=False); ax.grid(axis="x", color=GRID, lw=0.7, zorder=0)
fig.suptitle(f"Checkpoint comparison — {len(REQUESTS)} held-out queries",
             fontsize=12.5, color=INK, x=0.02, ha="left", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.955)); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3), facecolor="white")

ax = axes[0]
for v in MODELS:
    c = CRPS_CURVE[v].mean(axis=0)
    ax.plot(np.arange(1, len(c) + 1), c, color=MC[v], lw=2, zorder=3)
    ax.text(len(c) + 0.6, c[-1], v, color=MC[v], fontsize=9, va="center")
ax.set_xlim(1, HORIZON + 5); ax.set_xlabel("horizon step (candles ahead)", fontsize=9, color=MUTED)
ax.set_title("CRPS by horizon step — how fast each model's edge decays",
             fontsize=10.5, color=INK, loc="left", pad=10)
style(ax)

ax = axes[1]
grid = np.linspace(0, 1, 101)
for j, v in enumerate(MODELS):     # each curve labelled at its own x, so labels never stack
    pit = np.sort(M[v]["pit_terminal"])
    ecdf = np.searchsorted(pit, grid, side="right") / len(pit)
    xl = 0.30 + 0.15 * j
    yl = float(np.interp(xl, grid, ecdf))
    ax.plot(grid, ecdf, color=MC[v], lw=2, zorder=3)
    ax.plot([xl], [yl], "o", ms=7, color=MC[v], mec="white", mew=1.5, zorder=4)
    ax.text(xl + 0.022, yl - 0.062, v, color=MC[v], fontsize=9, va="center", ha="left", zorder=5)
ax.plot([0, 1], [0, 1], color=MUTED, lw=1.2, ls=(0, (4, 3)), zorder=2)
ax.text(0.62, 0.5, "perfect calibration", fontsize=8, color=MUTED, rotation=33)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel("nominal quantile of the terminal return", fontsize=9, color=MUTED)
ax.set_title("Calibration — empirical CDF of the realised outcome's rank in the fan",
             fontsize=10.5, color=INK, loc="left", pad=10)
style(ax)
fig.tight_layout(); plt.show()

In [ ]:
# fan charts: one panel per model on the same query
SHOW = 0                     # index into REQUESTS — change to inspect another query
sym, endi = KEPT[SHOW]
ts, oh = SYMS[sym]
hist = oh[endi - min(QUERY_LEN, 40) + 1:endi + 1]
actual = oh[endi + 1:endi + 1 + HORIZON]

def draw_candles(ax, x0, ohlc, up, down, width=0.62):
    for t, (o, h, l, c) in enumerate(ohlc):
        col = up if c >= o else down
        ax.plot([x0 + t, x0 + t], [l, h], color=col, lw=0.8, zorder=2)
        ax.add_patch(Rectangle((x0 + t - width / 2, min(o, c)), width,
                               max(abs(c - o), 1e-12), facecolor=col, edgecolor="none", zorder=3))

fig, axes = plt.subplots(2, 2, figsize=(12.5, 7), facecolor="white", sharey=True)
H = len(hist); fx = np.arange(H, H + HORIZON)
for ax, v in zip(axes.ravel(), MODELS):
    closes = FANS[v][SHOW][:, :, 3]
    q05, q25, q50, q75, q95 = np.percentile(closes, [5, 25, 50, 75, 95], axis=0)
    draw_candles(ax, 0, hist, "#9b9a95", "#9b9a95")     # shared history: neutral, not model-coloured
    ax.fill_between(fx, q05, q95, color=tint(MC[v], 0.82), zorder=1)
    ax.fill_between(fx, q25, q75, color=tint(MC[v], 0.6), zorder=1)
    ax.plot(fx, q50, color=MC[v], lw=2, zorder=4)
    top = SCEN[v][SHOW][0]
    ax.plot(fx, top.path, color=MC[v], lw=1.4, ls=(0, (3, 2)), zorder=5)
    ax.plot(fx, actual[:, 3], color=INK, lw=2, zorder=6)
    ax.axvline(H - 0.5, color=GRID, lw=1)
    ax.set_title(f"{v}   median · 5–95% fan · lead scenario (dashed) · realised (black)",
                 fontsize=9.5, color=INK, loc="left", pad=8)
    ax.text(0.015, 0.05, f"DTW {top.dtw:.3f}   best RMSE {min(s.rmse for s in SCEN[v][SHOW]):.3f}",
            transform=ax.transAxes, fontsize=8.5, color=MUTED)
    style(ax)
fig.suptitle(f"{sym} — query ending {utc(ts[endi]):%Y-%m-%d %H:%M} "
             f"(held-out query #{SHOW})", fontsize=12, color=INK, x=0.02, ha="left", y=0.99)
fig.tight_layout(rect=(0, 0, 1, 0.955)); plt.show()

## 10 · Optional — sensitivity to K and query length

The comparison above runs at `K = 50` / `query = 50` (what `val_k` and the scanner's midpoint look
like). A checkpoint that wins only at one operating point isn't the one to ship, so this sweeps
both. **Set `RUN_SWEEP = True`** — it costs roughly `len(SWEEP_K) + len(SWEEP_W)` times a fraction
of the main run (`SWEEP_N` queries each).

In [ ]:
RUN_SWEEP = False
SWEEP_N   = 20                     # queries per operating point
SWEEP_K   = [10, 50, 200]
SWEEP_W   = [20, 50, 100]

if RUN_SWEEP:
    sweep = {}
    subset = list(range(min(SWEEP_N, len(REQUESTS))))

    for K in SWEEP_K:                                  # K sweep reuses the K_SCAN retrieval
        for v in MODELS: sweep[(f"K={K}", v)] = []
        for i in subset:
            s, e = KEPT[i]
            req = make_request(s, e, ANALOGS[QUERIES.index((s, e))], K=K)
            for v in MODELS:
                r = run_generation(req, v, seed=SEED + i)
                sweep[(f"K={K}", v)].append(compute_metrics(r)[0])
        print(f"K={K} done")

    for W in SWEEP_W:                                  # query length needs its own scan
        qs = [KEPT[i] for i in subset]
        an = scan(qs, W, HORIZON, N_ANALOGS, EXCLUDE_SELF_BARS, verbose=False)
        for v in MODELS: sweep[(f"W={W}", v)] = []
        for (s, e), a, i in zip(qs, an, subset):
            req = make_request(s, e, a, W=W)
            for v in MODELS:
                r = run_generation(req, v, seed=SEED + i)
                sweep[(f"W={W}", v)].append(compute_metrics(r)[0])
        print(f"W={W} done")

    for metric in ("crps_step", "dtw_top1", "nll"):
        print(f"\nmedian {metric} by operating point (* = best)\n")
        rows = []
        for point in [f"K={k}" for k in SWEEP_K] + [f"W={w}" for w in SWEEP_W]:
            vals = {v: float(np.median([m[metric] for m in sweep[(point, v)]])) for v in MODELS}
            b = min(vals, key=vals.get)
            rows.append([point] + [f"{vals[v]:.4f}" + (" *" if v == b else "  ") for v in MODELS])
        print_table(["operating point"] + MODELS, rows)
else:
    print("sweep skipped — set RUN_SWEEP = True to run it")

## 11 · Skill — is the model better than not having one?

Everything above is relative: it says *which* checkpoint is best, never whether the best one is
**worth running**. That needs baselines the model has to beat. Four, from hardest to easiest:

| baseline | what it is | why it matters |
|---|---|---|
| **analog ensemble (vol-matched)** | the K retrieved reaction paths themselves, each rescaled by `s_query / s_analog` and re-anchored — no neural net at all | this is the model's actual competition. The scanner already produces it for free, so the model only earns its place by beating it |
| **analog ensemble (raw)** | the same paths without volatility matching | isolates how much of any edge is just vol normalization |
| **bootstrap random walk** | `NUM_SAMPLES` paths of i.i.d. resampled, de-meaned log returns from the query window | the classic no-skill null: right volatility, zero information about direction or shape |
| **flat** | one path that never moves | the floor. Losing to this would mean the generated moves are worse than predicting nothing |

Scored by the same `fan_metrics` code, with the **fair** CRPS/energy estimator so a 50-member
analog ensemble, a 64-path fan and a 1-path forecast are compared honestly. Skill score is
`1 − CRPS_model / CRPS_baseline` on pooled totals, with a 2 000-resample bootstrap CI over queries:
**positive means better than the baseline, and the CI has to clear 0 to mean anything.**

In [ ]:
def analog_ensemble(req, vol_match=True):
    # The retrieved analogs' own futures used directly as a forecast ensemble — the no-model
    # control. Each analog's reaction becomes a log-return path from its own last close,
    # optionally rescaled by the query/analog volatility ratio (the same vol_scale the model
    # divides by internally), then re-anchored on the query's last close.
    prep = FN.build_inputs(req, _cfg)
    A = float(req.q_anchor)
    s_q = vol_scale(ohlc_to_features(prep["q_pat"]), _cfg["vol_floor"])
    out = []
    for ap, af in zip(prep["patterns"], prep["futures"]):
        af = np.asarray(af, np.float64)[:HORIZON]
        if af.shape[0] < HORIZON: continue
        s_a = vol_scale(ohlc_to_features(np.asarray(ap, np.float64)), _cfg["vol_floor"])
        k = (s_q / s_a) if vol_match else 1.0
        out.append(A * np.exp(np.log(af[:, 3] / float(ap[-1, 3])) * k))
    return np.asarray(out)

def bootstrap_rw(req, n, rng):
    # De-meaned i.i.d. bootstrap of the query's own log returns: correct vol, zero signal.
    A = float(req.q_anchor)
    c = FN.candles_to_ohlc(req.scanned_candles)[:, 3]
    r = np.diff(np.log(c))
    r = r - r.mean()
    return A * np.exp(np.cumsum(rng.choice(r, size=(n, HORIZON), replace=True), axis=1))

BASE_SPECS = [
    ("analog ensemble (vol-matched)", lambda req, rng: analog_ensemble(req, True)),
    ("analog ensemble (raw)",         lambda req, rng: analog_ensemble(req, False)),
    ("bootstrap random walk",         lambda req, rng: bootstrap_rw(req, NUM_SAMPLES, rng)),
    ("flat (no change)",              lambda req, rng: np.full((1, HORIZON), float(req.q_anchor))),
]
BASELINES = [n for n, _ in BASE_SPECS]

rng_b = np.random.default_rng(SEED)
_acc = {n: [] for n in BASELINES}
for req in REQUESTS:
    actual = FN.candles_to_ohlc(req.forward_candle_data)[:, 3]
    A = float(req.q_anchor)
    for name, fn in BASE_SPECS:
        _acc[name].append(fan_metrics(fn(req, rng_b), actual, A)[0])
BM = {n: {k: np.array([m[k] for m in _acc[n]]) for k in _acc[n][0]} for n in BASELINES}
print("baseline ensemble sizes: analogs",
      len(analog_ensemble(REQUESTS[0])), "| bootstrap", NUM_SAMPLES, "| flat 1")

SHOWN = ["crps_step", "energy", "rmse_median", "cover90", "cover50", "dir_med"]
print(f"\nAbsolute scores over {len(REQUESTS)} held-out queries (median; rates are means)\n")
rows = []
for label, src in ([(v, M[v]) for v in MODELS] + [(b, BM[b]) for b in BASELINES]):
    f = lambda k: (np.mean if k in ("cover90", "cover50", "dir_med") else np.median)(src[k])
    rows.append([label] + [f"{f(k):.4f}" for k in SHOWN])
print_table(["forecaster"] + SHOWN, rows)
print("\n(the four models sit above the line, the four no-model controls below)")

In [ ]:
def skill_and_ci(model_vals, base_vals, n_boot=2000, seed=SEED):
    # Skill score 1 - sum(model)/sum(base), with a bootstrap CI over queries. Pooled totals
    # rather than a mean of per-query ratios: one query where the baseline scores near zero
    # would otherwise dominate the average.
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(model_vals), size=(n_boot, len(model_vals)))
    boot = 1.0 - model_vals[idx].sum(1) / base_vals[idx].sum(1)
    return (1.0 - model_vals.sum() / base_vals.sum(), *np.percentile(boot, [2.5, 97.5]))

for metric in ("crps_step", "energy"):
    print(f"SKILL vs each baseline on {metric} — skill%  [95% CI]  (win rate)\n")
    rows = []
    for b in BASELINES:
        row = [b]
        for v in MODELS:
            sk, lo, hi = skill_and_ci(M[v][metric], BM[b][metric])
            win = np.mean(M[v][metric] < BM[b][metric])
            row.append(f"{sk:+.1%} [{lo:+.0%},{hi:+.0%}] {win:.0%}")
        rows.append(row)
    print_table(["baseline"] + MODELS, rows)
    print()

print(f"VERDICT for the best checkpoint ({winner})\n")
for b in BASELINES:
    sk, lo, hi = skill_and_ci(M[winner][PRIMARY_METRIC], BM[b][PRIMARY_METRIC])
    beats = "BEATS" if lo > 0 else ("LOSES TO" if hi < 0 else "ties with")
    print(f"  {beats:9s} {b:32s} skill {sk:+.1%}  CI [{lo:+.1%}, {hi:+.1%}]")
sk, lo, hi = skill_and_ci(M[winner][PRIMARY_METRIC], BM[BASELINES[0]][PRIMARY_METRIC])
print("\n" + ("=> the model adds real information over just replaying the retrieved analogs."
               if lo > 0 else
               "=> the model does NOT clear the retrieval baseline on this sample: the analogs the\n"
               "   scanner already returns are as good or better, so the network is not yet earning\n"
               "   its place. Raise N_QUERIES to confirm before acting on it."))

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.6), facecolor="white")
y = np.arange(len(BASELINES))
for j, v in enumerate(MODELS):
    off = (j - (len(MODELS) - 1) / 2) * 0.17
    xs, los, his = [], [], []
    for b in BASELINES:
        sk, lo, hi = skill_and_ci(M[v][PRIMARY_METRIC], BM[b][PRIMARY_METRIC])
        xs.append(sk * 100); los.append(sk * 100 - lo * 100); his.append(hi * 100 - sk * 100)
    ax.errorbar(xs, y + off, xerr=[los, his], fmt="o", ms=8, color=MC[v], ecolor=tint(MC[v], 0.55),
                elinewidth=2, capsize=0, label=v, zorder=3)
ax.axvline(0, color=INK, lw=1.4, zorder=2)
ax.text(0.6, len(BASELINES) - 0.35, "no better than the baseline →  0", fontsize=8, color=MUTED)
ax.set_yticks(y); ax.set_yticklabels(BASELINES, fontsize=9.5, color=INK); ax.invert_yaxis()
ax.set_xlabel(f"skill score on {PRIMARY_METRIC} (%, higher is better) — bars are 95% bootstrap CIs",
              fontsize=8.5, color=MUTED, loc="left")
ax.set_title("Does the network beat having no network?", fontsize=11.5, color=INK, loc="left", pad=10)
leg = ax.legend(loc="lower right", frameon=False, fontsize=9, ncol=len(MODELS), handletextpad=0.2)
for t, v in zip(leg.get_texts(), MODELS): t.set_color(MC[v])
style(ax, ygrid=False); ax.grid(axis="x", color=GRID, lw=0.7, zorder=0)
fig.tight_layout(); plt.show()

### 11b · Would a tighter sampler close the gap?

Every checkpoint's fan is wider than reality (`vol_ratio > 1`), and CRPS punishes excess spread —
so part of any shortfall against the baselines may be sampling temperature rather than a weak model.
This re-scores the chosen checkpoint at several `sigma_scale` values against the same baselines.
If skill only turns positive at `sigma_scale < 1`, the fix is a config change, not more training.

In [ ]:
RUN_SIGMA_SWEEP = False
SIGMA_MODEL = None                 # None = the winner from section 9
SIGMA_VALUES = [1.0, 0.85, 0.7, 0.55]
SIGMA_N = 40                       # queries per setting

if RUN_SIGMA_SWEEP:
    mdl = SIGMA_MODEL or winner
    ref = ["analog ensemble (vol-matched)", "bootstrap random walk"]
    sub = list(range(min(SIGMA_N, len(REQUESTS))))
    _sigma_keep = SIGMA_SCALE
    rows = []
    for ss in SIGMA_VALUES:
        SIGMA_SCALE = ss                       # run_generation reads this global
        acc = []
        for i in sub:
            r = run_generation(REQUESTS[i], mdl, seed=SEED + i)
            acc.append(fan_metrics(r["paths"], r["actual"][:, 3], r["anchor"])[0])
        mm = {k: np.array([a[k] for a in acc]) for k in acc[0]}
        row = [f"{ss:.2f}", f"{np.median(mm['crps_step']):.5f}", f"{mm['cover90'].mean():.3f}",
               f"{mm['cover50'].mean():.3f}", f"{np.median(mm['vol_ratio']):.3f}"]
        for b in ref:
            sk, lo, hi = skill_and_ci(mm["crps_step"], BM[b]["crps_step"][sub])
            row.append(f"{sk:+.1%} [{lo:+.0%},{hi:+.0%}]")
        rows.append(row)
        print(f"  sigma_scale {ss:.2f} done")
    SIGMA_SCALE = _sigma_keep
    print(f"\n{mdl} across sigma_scale, {len(sub)} queries "
          f"(coverage targets 0.90 / 0.50, vol ratio 1.00)\n")
    print_table(["sigma", "CRPS", "cov90", "cov50", "vol ratio",
                 "skill vs analogs", "skill vs RW"], rows)
else:
    print("sigma sweep skipped — set RUN_SIGMA_SWEEP = True to run it")

## 12 · How to read this, and what it does not say

* **Section 11 is the one that decides whether to ship anything.** Sections 8–10 only rank the
  four checkpoints against each other; a model can win that contest and still be worthless. The
  skill scores against the analog ensemble are the pass/fail: the scanner hands you those paths for
  free, so a checkpoint that does not clear them is adding compute, not information.
* **`nll` is the cleanest single verdict between checkpoints.** It is the training objective evaluated on futures no
  checkpoint was trained on, converted into common units, and it has no sampling noise. `crps_step`
  and `energy` are the sampling-based counterparts and should agree with it; where they disagree,
  the model is producing a good density but a poorly-calibrated *sample* (check `cover90` and
  `vol_ratio`).
* **The DTW columns are oracle-selected on purpose.** With `forward_candle_data` present,
  `functions.py` ranks the fan against the realised path — so `dtw_top1` answers *"was the right
  movement anywhere in the fan"*, not *"would the model have picked it live"*. A model can win on
  DTW simply by sampling wider. Read it next to `cover90` and `vol_ratio`: an over-wide fan shows
  up as coverage well above 0.90 and a vol ratio above 1.
* **Coverage is a target, not a race.** 0.90 is right; 0.99 means the fan is too wide to be useful,
  0.60 means it is overconfident.
* **`3y`, `7y` and `all` were stopped early** (epochs 7, 11, 8 of 40) while `1y` finished 40/40.
  Any gap you see is a training-budget gap as much as a data-recipe gap — it does not say the `1y`
  analog filter is intrinsically better.
* **Hold-out is positional, not temporal.** All four datasets were drawn from these same CSVs
  across the same 2010–2026 span, so there is no period that is clean by construction. The
  ±150-bar exclusion guarantees no *scored future* overlaps a *trained target*, but the same symbol
  and regime were seen elsewhere in training. That is as clean as this data allows.
* **The scanner is a faithful re-implementation, not the live service.** Cosine parity is ~1e-6,
  but the production bank may cover different symbols or a different stride, so retrieved analogs
  can differ from what the service would return for the same bar.
* **All four fans run wide at `sigma_scale = 1.0`** — the vol ratio sits above 1 for every
  checkpoint, so 90% bands swallow nearly every realised path. That is a sampling knob, not a
  ranking artifact: re-run with `SIGMA_SCALE = 0.7–0.8` to see how each model scores once the
  spread is tightened, and note it shifts CRPS/energy for all of them.
* Raising `N_QUERIES` narrows the error on every number here; 60 queries resolves large gaps
  reliably, and 200+ is worth it before making a shipping decision on a close call.